In [8]:
"""
Nepali NLP Tokenization Pipeline
=================================
Steps performed:
  1. Whitespace tokenization
  2. Token type tagging  (devanagari / digit / latin / mixed / punctuation)
  3. Token frequency computation
  4. Stopword candidate detection  (high-freq function words)
  5. Suffix inventory              (for future stemming / stripping)
  6. Vocabulary index (word → id, id → word)

Outputs
-------
  tokenized_dataset.csv   — one row per original sentence, tokens as JSON list
  token_freq.csv          — token, count, type, is_stopword_candidate
  suffix_inventory.csv    — suffix, count
  vocab.json              — {word2id, id2word, stopword_candidates, suffix_inventory}
"""

import csv, json, re, collections, unicodedata
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_CSV   = "normalized_dataset.csv"

# Nepali grammatical suffixes to detect (longest first for greedy match)
NEPALI_SUFFIXES = [
    "बाटनै","दएखिनै","कएबल",
    "कओसग","लाइनै",
    "कओ","एकओ","नए","लए","लाइ","मा","पछि","दएखि","बाट","सग",
    "हरु","लाइ","मात्र","सम्म","भित्र","बाहिर","अगाडि","पछाडि",
    "माथि","तल","नजिक","टाढा","साथ","बिना","लागि","कारण",
]
NEPALI_SUFFIXES.sort(key=len, reverse=True)

# Devanagari range
DEVA_RE    = re.compile(r'^[\u0900-\u097F\u0966-\u096F।]+$')
DIGIT_RE   = re.compile(r'^[\d०-९.]+$')
LATIN_RE   = re.compile(r'^[a-zA-Z]+$')

# Known Nepali stopwords / function words (post-normalization spellings)
KNOWN_STOPWORDS = {
    "र","छ","छऐन","तर","पनि","कम","अलि","मा","को","का","की",
    "यो","त्यो","हो","हैन","गर्न","गरि","गरेको","भएको","भएर",
    "हुन","हुने","हुन्छ","सो","जो","जस्तो","यस्तो","अनि","वा",
    "नि","नै","पनि","सबै","कुनै","केही","धेरै","थोरै","एक",
    "दुइ","तिन","चार","पाँच","भन्दा","बाट","सग","लाइ","लागि",
    # normalized forms
    "छऐन","भएकओ","हुनए","गरिएकओ","उनकओ","उसकओ","धएरऐ","थओरऐ",
    "जस्तओ","कुनऐ","राम्रओ","नानिलए","दिनकओ","दएखिएकओ",
}

In [9]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def token_type(tok: str) -> str:
    if DIGIT_RE.match(tok):     return "digit"
    if LATIN_RE.match(tok):     return "latin"
    if DEVA_RE.match(tok):      return "devanagari"
    if re.search(r'[a-zA-Z\d]', tok): return "mixed"
    return "other"


def get_suffix(tok: str) -> str | None:
    """Return the longest matching suffix, or None."""
    if len(tok) <= 2:
        return None
    for sfx in NEPALI_SUFFIXES:
        if tok.endswith(sfx) and len(tok) > len(sfx) + 1:
            return sfx
    return None


def tokenize(text: str) -> list[str]:
    """
    Simple whitespace tokenizer.
    Splits on spaces; also splits tokens that still contain the em-dash (–)
    left by normalization (e.g. सानि–सानि → ['सानि', 'सानि']).
    """
    raw = text.split()
    tokens = []
    for t in raw:
        # split on em-dash or hyphen that survived normalization
        parts = re.split(r'[–\-]', t)
        tokens.extend(p for p in parts if p)
    return tokens

In [10]:
# ── Main pipeline ─────────────────────────────────────────────────────────────

all_token_lists: list[list[str]] = []
rows_in: list[dict] = []

with open(INPUT_CSV, encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        rows_in.append(row)
        all_token_lists.append(tokenize(row["normalized_text"]))

# ── 1. Tokenized dataset ──────────────────────────────────────────────────────
tok_csv = "tokenized_dataset.csv"
with open(tok_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["original_text","normalized_text","tokens","label"])
    w.writeheader()
    for row, toks in zip(rows_in, all_token_lists):
        w.writerow({
            "original_text":  row["original_text"],
            "normalized_text": row["normalized_text"],
            "tokens":         json.dumps(toks, ensure_ascii=False),
            "label":          row["label"],
        })

print(f"✅  tokenized_dataset.csv  — {len(rows_in)} rows")

# ── 2. Token frequency + type + stopword flag ─────────────────────────────────
flat_tokens = [t for toks in all_token_lists for t in toks]
freq = collections.Counter(flat_tokens)
total = len(flat_tokens)

# Auto-detect high-frequency stopword candidates:
# top 5 % by frequency that are pure Devanagari and short (≤ 6 chars)
freq_threshold = sorted(freq.values(), reverse=True)[max(1, len(freq)//20)]

stopword_candidates: set[str] = set()
for tok, cnt in freq.items():
    is_known   = tok in KNOWN_STOPWORDS
    is_highfreq = (
        cnt >= freq_threshold
        and token_type(tok) == "devanagari"
        and len(tok) <= 8
    )
    if is_known or is_highfreq:
        stopword_candidates.add(tok)

freq_csv = "token_freq.csv"
with open(freq_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["token","count","pct_of_corpus","type","is_stopword_candidate"])
    w.writeheader()
    for tok, cnt in freq.most_common():
        w.writerow({
            "token":                tok,
            "count":                cnt,
            "pct_of_corpus":        f"{cnt/total*100:.3f}",
            "type":                 token_type(tok),
            "is_stopword_candidate": int(tok in stopword_candidates),
        })

print(f"✅  token_freq.csv         — {len(freq)} unique tokens, "
      f"{len(stopword_candidates)} stopword candidates")

# ── 3. Suffix inventory ───────────────────────────────────────────────────────
suffix_counts: dict[str, int] = collections.Counter()
suffix_examples: dict[str, list[str]] = collections.defaultdict(list)

for tok in flat_tokens:
    sfx = get_suffix(tok)
    if sfx:
        suffix_counts[sfx] += 1
        if len(suffix_examples[sfx]) < 5 and tok not in suffix_examples[sfx]:
            suffix_examples[sfx].append(tok)

sfx_csv = "suffix_inventory.csv"
with open(sfx_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["suffix","count","example_tokens"])
    w.writeheader()
    for sfx, cnt in sorted(suffix_counts.items(), key=lambda x: -x[1]):
        w.writerow({
            "suffix":         sfx,
            "count":          cnt,
            "example_tokens": " | ".join(suffix_examples[sfx]),
        })

print(f"✅  suffix_inventory.csv   — {len(suffix_counts)} suffix types")

# ── 4. Vocabulary JSON (word2id / id2word + metadata) ────────────────────────
# Reserve 0 = <PAD>, 1 = <UNK>
word2id = {"<PAD>": 0, "<UNK>": 1}
for tok, _ in freq.most_common():
    word2id[tok] = len(word2id)

id2word = {v: k for k, v in word2id.items()}

vocab_json = "vocab.json"
with open(vocab_json, "w", encoding="utf-8") as f:
    json.dump({
        "vocab_size":           len(word2id),
        "total_tokens":         total,
        "unique_tokens":        len(freq),
        "word2id":              word2id,
        "id2word":              {str(k): v for k, v in id2word.items()},
        "stopword_candidates":  sorted(stopword_candidates),
        "suffix_inventory":     dict(sorted(suffix_counts.items(), key=lambda x: -x[1])),
    }, f, ensure_ascii=False, indent=2)

print(f"✅  vocab.json             — vocab size: {len(word2id)}")

# ── 5. Summary report ─────────────────────────────────────────────────────────
print()
print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"  Sentences processed  : {len(rows_in)}")
print(f"  Total tokens         : {total}")
print(f"  Unique tokens        : {len(freq)}")
print(f"  Stopword candidates  : {len(stopword_candidates)}")
print(f"  Suffix types found   : {len(suffix_counts)}")
print(f"  Vocab size (w/ spec) : {len(word2id)}")

print(f"\n  Top stopword candidates:")
for s in sorted(stopword_candidates, key=lambda x: -freq[x])[:20]:
    print(f"    {freq[s]:4d}  {s}")

print(f"\n  Top suffixes:")
for sfx, cnt in sorted(suffix_counts.items(), key=lambda x: -x[1])[:15]:
    print(f"    {cnt:4d}  -{sfx}   e.g. {suffix_examples[sfx][0]}")

print()

✅  tokenized_dataset.csv  — 705 rows
✅  token_freq.csv         — 1589 unique tokens, 94 stopword candidates
✅  suffix_inventory.csv   — 13 suffix types
✅  vocab.json             — vocab size: 1591

PIPELINE SUMMARY
  Sentences processed  : 705
  Total tokens         : 7209
  Unique tokens        : 1589
  Stopword candidates  : 94
  Suffix types found   : 13
  Vocab size (w/ spec) : 1591

  Top stopword candidates:
     427  र
     410  छ
     134  छऐन
     116  तर
      92  सामान्य
      91  हल्का
      82  उनकओ
      79  कम
      72  उसकओ
      71  सास
      68  थओरऐ
      65  भएकओ
      64  पनि
      63  अलि
      58  बर्स
      58  कुनऐ
      57  हप्ता
      55  दुध
      54  दुखाइ
      51  धएरऐ

  Top suffixes:
     554  -कओ   e.g. गर्भाबस्थाकओ
     250  -एकओ   e.g. बगएकओ
     191  -मा   e.g. बरिद्धिमा
     189  -नए   e.g. आउनए
     139  -लए   e.g. गर्भाबस्थालए
      72  -लाइ   e.g. उनलाइ
      33  -पछि   e.g. पिएपछि
      26  -दएखि   e.g. बिहानदएखि
      17  -बाट   e.g. सरिरबाट
 